In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("PySpark Basic Algorithms") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("==============================================")
print("       PySpark Basic Algorithms")
print("==============================================")


# =========================================================
# 1. MATRIX MULTIPLICATION
# =========================================================

print("\n========== Matrix Multiplication ==========")

n = int(input("Enter matrix size (2 for 2x2): "))

print("\nEnter Matrix A elements row-wise:")

A = []

for i in range(n):
    for j in range(n):
        value = int(input(f"A[{i}][{j}]: "))
        A.append((i, j, value))


print("\nEnter Matrix B elements row-wise:")

B = []

for i in range(n):
    for j in range(n):
        value = int(input(f"B[{i}][{j}]: "))
        B.append((i, j, value))


rddA = sc.parallelize(A)
rddB = sc.parallelize(B)


# A: (i,j,value) -> (j,(i,value))
A_pairs = rddA.map(
    lambda x: (x[1], (x[0], x[2]))
)

# B: (j,k,value) -> (j,(k,value))
B_pairs = rddB.map(
    lambda x: (x[0], (x[1], x[2]))
)


result_matrix = (
    A_pairs
    .join(B_pairs)
    .map(lambda x: (
        (x[1][0][0], x[1][1][0]),
        x[1][0][1] * x[1][1][1]
    ))
    .reduceByKey(lambda x, y: x + y)
)


print("\nMatrix Multiplication Result:")

for cell in sorted(result_matrix.collect()):
    print(cell)


# =========================================================
# 2. AGGREGATE
# =========================================================

print("\n========== Aggregate ==========")

count = int(input("Enter number of sales records: "))

sales = []

for i in range(count):
    dept = input("Enter department: ")
    amount = int(input("Enter amount: "))
    sales.append((dept, amount))


salesRDD = sc.parallelize(sales)

aggregate = salesRDD.reduceByKey(
    lambda x, y: x + y
)


print("\nTotal Sales:")

for item in aggregate.collect():
    print(item)


# =========================================================
# 3. JOIN
# =========================================================

print("\n========== Join ==========")

students = []
marks = []

n = int(input("Enter number of students: "))

for i in range(n):

    roll = int(input("Enter roll number: "))
    name = input("Enter name: ")
    mark = int(input("Enter mark: "))

    students.append((roll, name))
    marks.append((roll, mark))


studentRDD = sc.parallelize(students)
marksRDD = sc.parallelize(marks)


join_result = studentRDD.join(marksRDD)


print("\nStudent Details:")

for item in join_result.collect():
    print(item)


# =========================================================
# 4. SORTING
# =========================================================

print("\n========== Sorting ==========")

nums = input(
    "Enter numbers separated by space: "
)

numbers = list(
    map(int, nums.split())
)


numberRDD = sc.parallelize(numbers)

sorted_numbers = numberRDD.sortBy(
    lambda x: x
)


print("\nSorted Numbers:")
print(sorted_numbers.collect())


# =========================================================
# 5. SEARCHING
# =========================================================

print("\n========== Searching ==========")

lines = []

n = int(input("Enter number of sentences: "))

for i in range(n):

    sentence = input(
        f"Enter sentence {i + 1}: "
    )

    lines.append(sentence)


word = input("Enter word to search: ")


textRDD = sc.parallelize(lines)


search_result = textRDD.filter(
    lambda x: word.lower() in x.lower()
)


print("\nSearch Result:")

for line in search_result.collect():
    print(line)


# =========================================================
# 6. UNION, INTERSECTION AND DIFFERENCE
# =========================================================

print("\n========== Union, Intersection and Difference ==========")


set1 = list(
    map(
        int,
        input(
            "Enter elements of Set 1 separated by space: "
        ).split()
    )
)


set2 = list(
    map(
        int,
        input(
            "Enter elements of Set 2 separated by space: "
        ).split()
    )
)


rdd1 = sc.parallelize(set1)
rdd2 = sc.parallelize(set2)


# Union
union_result = rdd1.union(rdd2).distinct()

# Intersection
intersection_result = rdd1.intersection(rdd2)

# Difference
difference_result = rdd1.subtract(rdd2)

print("\nUnion:")
print(sorted(union_result.collect()))


print("\nIntersection:")
print(sorted(intersection_result.collect()))


print("\nDifference (Set1 - Set2):")
print(sorted(difference_result.collect()))


# =========================================================
# STOP SPARK
# =========================================================

sc.stop()

print("\n==============================================")
print("        Program Completed Successfully")
print("==============================================")

       PySpark Basic Algorithms

========== Matrix Multiplication ==========
Enter matrix size (2 for 2x2): 2

Enter Matrix A elements row-wise:
A[0][0]: 44
A[0][1]: 55
A[1][0]: 4545
A[1][1]: 5452

Enter Matrix B elements row-wise:
B[0][0]: 6656
B[0][1]: 534
B[1][0]: 23455
B[1][1]: 454356

Matrix Multiplication Result:
((0, 0), 1582889)
((0, 1), 25013076)
((1, 0), 158128180)
((1, 1), 2479575942)

========== Aggregate ==========
Enter number of sales records: 2
Enter department: aids
Enter amount: 80000
Enter department: it
Enter amount: 85000

Total Sales:
('aids', 80000)
('it', 85000)

========== Join ==========
Enter number of students: 2
Enter roll number: 05
Enter name: swayam
Enter mark: 99
Enter roll number: 21
Enter name: shridhar
Enter mark: 100

Student Details:
(5, ('swayam', 99))
(21, ('shridhar', 100))

========== Sorting ==========
Enter numbers separated by space: 3 55 64 45 45 545 4543535 0

Sorted Numbers:
[0, 3, 45, 45, 55, 64, 545, 4543535]

========== Searching =====